In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV

# 1. Feature Separation
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Preprocessing Pipelines
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

# 4. Model Pipeline
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA()),
    ('model', LogisticRegression(max_iter=2000))
])

# 5. Hyperparameter Tuning
params = {
    'pca__n_components': [2, 3, 4],
    'model__C': [0.1, 1, 10]
}

grid = GridSearchCV(
    pipe, params, cv=5, scoring='accuracy', n_jobs=-1
)
grid.fit(X_train, y_train)

# 6. Predictions
pred = grid.best_estimator_.predict(X_test)
proba = grid.best_estimator_.predict_proba(X_test)[:, 1]  # Required for ROC-AUC

# ==============================================================
# 📊 PERFORMANCE METRICS (CLASSIFICATION)
# ==============================================================
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_auc_score, roc_curve
)

# 1. Core Summary Metrics
print("=== Classification Metrics ===")
print(f"Accuracy:  {accuracy_score(y_test, pred):.4f}")
print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall:    {recall_score(y_test, pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, proba):.4f}")

# 2. Detailed Classification Report
print("\n=== Classification Report ===")
print(classification_report(y_test, pred))

# 3. Confusion Matrix and ROC Curve Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion Matrix Display
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap='Blues', ax=axes[0])
axes[0].set_title(f"Confusion Matrix (Acc = {accuracy_score(y_test, pred):.2%})")
axes[0].grid(False)

# ROC Curve Plot
fpr, tpr, _ = roc_curve(y_test, proba)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f"ROC (AUC = {roc_auc_score(y_test, proba):.3f})")
axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()




# ==============================================================
# 📈 PERFORMANCE METRICS (REGRESSION)
# ==============================================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predictions
pred = grid.best_estimator_.predict(X_test)

mae = mean_absolute_error(y_test, pred)
mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)

# Adjusted R-squared
n = len(y_test)
p = X_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("=== Regression Performance Metrics ===")
print(f"MAE (Mean Absolute Error):     {mae:.4f}")
print(f"MSE (Mean Squared Error):      {mse:.4f}")
print(f"RMSE (Root Mean Squared Error):{rmse:.4f}")
print(f"R² Score:                      {r2:.4f}")
print(f"Adjusted R²:                   {adj_r2:.4f}")

# Actual vs Predicted Scatter Plot
plt.figure(figsize=(6, 5))
plt.scatter(y_test, pred, color='royalblue', alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title(f"Actual vs Predicted (R² = {r2:.3f})")
plt.show()
